# Group 3 — Boxer shuffle (music vs no music)

**Trial 1's acceleration was clipped — use Trial 2.** Look at the calf muscle (EMG) and the shank bounce (accel), then compare with-music vs without.

Your sensors: **Ch 1 = R tibia**, **Ch 2 = R medial gastroc**, **Ch 3 = L medial gastroc**, **Ch 4 = L tibia** — each Delsys sensor gives EMG **and** acceleration.

In [ ]:
!pip install -q delsys pysampled huggingface_hub

### 1 · Get your data (Trial 2 — Trial 1 was clipped)

In [ ]:
from huggingface_hub import hf_hub_download
REPO = "praneethnamburimit/immersionlab-pe-mis-groups"
csv = hf_hub_download(REPO, "group_3/Trial_2.csv", repo_type="dataset")
print(csv)

### 2 · Load it

In [ ]:
import delsys
lf = delsys.Log(csv)
lf

### 3 · Calf muscle (EMG) + shank bounce (accel)

In [ ]:
import matplotlib.pyplot as plt
gastroc = lf.find(sensor_number=2, as_="sensor")[0].emg   # Ch 2 = R medial gastroc
tibia   = lf.find(sensor_number=1, as_="sensor")[0].acc   # Ch 1 = R tibia (acceleration)
fig, ax = plt.subplots(2,1, figsize=(12,5), sharex=True)
ax[0].plot(gastroc.t, gastroc.bandpass(20,450).envelope()); ax[0].set_title("medial gastroc — muscle effort")
ax[1].plot(tibia.t, tibia()); ax[1].set_title("tibia — acceleration (x, y, z)")
ax[1].set_xlabel("s"); [a.set_xlim(15,20) for a in ax]; plt.show()

### 4 · Cadence from the shank accel

In [ ]:
import numpy as np
from scipy.signal import welch
acc = np.asarray(tibia()); mag = np.sqrt((acc**2).sum(axis=1))
f, P = welch(mag - mag.mean(), fs=tibia.sr, nperseg=int(tibia.sr*8))
b = (f >= 1) & (f <= 5); cad = f[b][np.argmax(P[b])]
print(f"cadence ~ {cad:.1f} steps/s = {cad*60:.0f} per minute")

### 5 · Your turn
- Split Trial 2 into the **music** and **no-music** shuffle windows (you know the order) and compare the calf **EMG effort** and the **cadence** between them — did music change either?
- The left leg is there too: `sensor_number=3` (gastroc), `sensor_number=4` (tibia).